# 07a — Dataset Baselines

Compare the two processed dataset versions with the current V1-style feature set using the main baseline frameworks, the two primary split strategies, and both target choices.

## 1) Imports & Configuration

Set up paths, shared experiment utilities, datasets to compare, and the artifact directory for this notebook.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'v2_experiment_utils.py').exists():
    NOTEBOOK_DIR = Path('/home/mo-seif/Documents/GP/ml-service/notebooks')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from v2_experiment_utils import (
    RANDOM_STATE,
    TARGET_COLS,
    DEFAULT_QUANTILES_3,
    XGB_BASE_PARAMS,
    LGBM_BASE_PARAMS,
    available_dataset_tags,
    load_processed_dataset,
    dataset_summary,
    build_primary_splits,
    build_feature_spec,
    make_lgbm_frame,
    fit_label_encoders,
    transform_with_label_encoders,
    split_train_val_indices,
    train_lgbm_quantile_models,
    train_xgb_quantile_models,
    predict_lgbm_quantiles,
    predict_xgb_quantiles,
    blend_prediction_maps,
    evaluate_point_and_interval,
    per_make_model_mape_df,
    run_kfold_per_make_model_cv,
    make_experiment_dir,
)

pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
np.set_printoptions(suppress=True)

DATASET_TAGS = ['2026-05-28_001', '2026-06-03_008']
PRIMARY_SPLITS = ['price_stratified', 'make_model_grouped']
QUANTILES = DEFAULT_QUANTILES_3
OUT_DIR = make_experiment_dir('07a_dataset_baselines')
OUT_DIR

PosixPath('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07a_dataset_baselines')

## 2) Dataset Selection & Output Paths

Confirm the processed dataset versions to compare and where baseline outputs will be written.

In [2]:
print('Selected dataset tags:', DATASET_TAGS)
print('Output directory:', OUT_DIR)

Selected dataset tags: ['2026-05-28_001', '2026-06-03_008']
Output directory: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07a_dataset_baselines


## 3) Load Datasets & Build Primary Splits

Load each processed snapshot, summarize it, and freeze the two primary split strategies for downstream notebook reuse.

In [3]:
dataset_store: dict[str, pd.DataFrame] = {}
split_store: dict[str, dict[str, tuple[np.ndarray, np.ndarray]]] = {}
summary_rows: list[pd.DataFrame] = []

for tag in DATASET_TAGS:
    df_tag = load_processed_dataset(tag)
    dataset_store[tag] = df_tag
    split_store[tag] = build_primary_splits(df_tag)
    summary = dataset_summary(df_tag)
    summary.insert(0, 'dataset_tag', tag)
    summary_rows.append(summary)

summary_df = pd.concat(summary_rows, ignore_index=True)
display(summary_df)

for tag, split_map in split_store.items():
    print(f'\nDataset {tag}')
    for split_name, (tr_idx, te_idx) in split_map.items():
        print(f'  {split_name:<20} train={len(tr_idx):,} test={len(te_idx):,}')

,dataset_tag,rows,make_count,model_count,price_min,price_median,price_max
0,2026-05-28_001,19451,68,487,22222,"630,000.0000",20000000
1,2026-06-03_008,12498,65,436,25000,"725,000.0000",20000000



Dataset 2026-05-28_001
  price_stratified     train=15,560 test=3,891
  make_model_grouped   train=16,548 test=2,903

Dataset 2026-06-03_008
  price_stratified     train=9,998 test=2,500
  make_model_grouped   train=10,371 test=2,127


## 4) Baseline Feature Specification

Use the current V1-style feature set exactly so the comparisons stay apples-to-apples.

In [4]:
NUM_COLS, CAT_COLS, FEATURE_COLS = build_feature_spec([])
print('Base numeric columns:', NUM_COLS)
print('Base categorical columns:', CAT_COLS)
print('Total features:', len(FEATURE_COLS))

Base numeric columns: ['year', 'mileage_km', 'mileage_per_year', 'engine_cc', 'horsepower', 'seating_capacity']
Base categorical columns: ['make', 'model', 'transmission', 'fuel', 'location', 'body_type', 'drivetrain', 'brand_origin', 'car_segment']
Total features: 15


## 5) Baseline Training Helper

Define the reusable training routine for one dataset, one split, and one target using XGBoost, LightGBM, and the weighted ensemble.

In [5]:
def fit_baseline_for_split(df_full: pd.DataFrame, train_idx: np.ndarray, test_idx: np.ndarray, target_col: str) -> tuple[dict, dict, pd.DataFrame]:
    is_log_target = target_col == 'price_egp_log'
    train_idx_fit, val_idx = split_train_val_indices(train_idx, val_fraction=0.10, random_state=RANDOM_STATE)

    train_df = df_full.loc[train_idx_fit].copy()
    val_df = df_full.loc[val_idx].copy()
    test_df = df_full.loc[test_idx].copy()

    X_train_lgb = make_lgbm_frame(train_df, FEATURE_COLS, CAT_COLS)
    X_val_lgb = make_lgbm_frame(val_df, FEATURE_COLS, CAT_COLS)
    X_test_lgb = make_lgbm_frame(test_df, FEATURE_COLS, CAT_COLS)

    label_encoders = fit_label_encoders(train_df, CAT_COLS)
    X_train_xgb = transform_with_label_encoders(train_df, FEATURE_COLS, CAT_COLS, label_encoders)
    X_val_xgb = transform_with_label_encoders(val_df, FEATURE_COLS, CAT_COLS, label_encoders)
    X_test_xgb = transform_with_label_encoders(test_df, FEATURE_COLS, CAT_COLS, label_encoders)

    lgbm_models = train_lgbm_quantile_models(
        X_train_lgb,
        train_df[target_col],
        X_val_lgb,
        val_df[target_col],
        CAT_COLS,
        QUANTILES,
        params=LGBM_BASE_PARAMS,
        n_estimators_max=1200,
        early_stopping_rounds=40,
    )
    xgb_models = train_xgb_quantile_models(
        X_train_xgb,
        train_df[target_col],
        X_val_xgb,
        val_df[target_col],
        QUANTILES,
        params=XGB_BASE_PARAMS,
        n_estimators_max=1200,
        early_stopping_rounds=40,
    )

    lgbm_pred_map = predict_lgbm_quantiles(lgbm_models, X_test_lgb)
    xgb_pred_map = predict_xgb_quantiles(xgb_models, X_test_xgb)
    ensemble_pred_map = blend_prediction_maps([xgb_pred_map, lgbm_pred_map], weights=[0.55, 0.45])

    framework_rows = []
    framework_maps = {
        'LightGBM': lgbm_pred_map,
        'XGBoost': xgb_pred_map,
        'WeightedEnsemble': ensemble_pred_map,
    }
    for framework, pred_map in framework_maps.items():
        metrics = evaluate_point_and_interval(
            test_df[target_col].to_numpy(),
            pred_map,
            is_log_target=is_log_target,
        )
        framework_rows.append({'framework': framework, **metrics})

    result_df = pd.DataFrame(framework_rows)
    best_framework = result_df.sort_values(['MAPE_pct', 'MAE']).iloc[0]['framework']
    median_key = 'median' if 'median' in framework_maps[best_framework] else 'q50'
    median_pred = np.asarray(framework_maps[best_framework][median_key], dtype=float)
    if is_log_target:
        if float(np.nanmedian(test_df[target_col].to_numpy())) < 10:
            median_pred = np.power(10.0, median_pred)
        else:
            median_pred = np.exp(median_pred)
    diag_df = per_make_model_mape_df(
        test_df,
        median_pred,
        min_rows=5,
    )
    metadata = {
        'models': {
            'lgbm': lgbm_models,
            'xgb': xgb_models,
        },
        'predictions': framework_maps,
        'test_df': test_df,
        'best_framework': best_framework,
        'label_encoders': label_encoders,
    }
    return metadata, result_df, diag_df

In [6]:
## 6) Run Best Baseline Only (fast path)

# Skip the full 8-run matrix and train only the selected best configuration to save time.

dataset_tag = '2026-06-03_008'
split_name = 'price_stratified'
target_col = 'price_egp_log'

df_full = load_processed_dataset(dataset_tag)
split_map = build_primary_splits(df_full)
train_idx, test_idx = split_map[split_name]

artifact, metrics_df, diag_df = fit_baseline_for_split(df_full, train_idx, test_idx, target_col)
metrics_df.insert(0, 'target_col', target_col)
metrics_df.insert(0, 'split_name', split_name)
metrics_df.insert(0, 'dataset_tag', dataset_tag)

baseline_results_df = metrics_df.copy()
baseline_diags = {(dataset_tag, split_name, target_col): diag_df}
baseline_artifacts = {(dataset_tag, split_name, target_col): artifact}

# Build minimal split_store / dataset_store for downstream export
dataset_store = {dataset_tag: df_full}
split_store = {dataset_tag: split_map}
summary_df = dataset_summary(df_full)
summary_df.insert(0, 'dataset_tag', dataset_tag)

print(f'Completed focused baseline run: {(dataset_tag, split_name, target_col)}')
display(baseline_results_df)

Completed focused baseline run: ('2026-06-03_008', 'price_stratified', 'price_egp_log')


,dataset_tag,split_name,target_col,framework,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Mean_width_pct
0,2026-06-03_008,price_stratified,price_egp_log,LightGBM,"154,151.0324","606,744.8389",0.8871,11.5839,60.8400,76.3200,79.0800,79.0800,48.4552
1,2026-06-03_008,price_stratified,price_egp_log,XGBoost,"154,313.7665","613,958.9999",0.8844,11.5368,60.0800,76.1600,87.4000,87.4000,73.1503
2,2026-06-03_008,price_stratified,price_egp_log,WeightedEnsemble,"150,593.8633","605,558.6220",0.8875,11.3258,61.2400,76.7600,85.7600,85.7600,61.5454


## 6) Run Baseline Matrix

Train and evaluate the baseline recipe across both dataset versions, both primary split strategies, and both target choices.

In [7]:
baseline_results: list[pd.DataFrame] = []
baseline_diags: dict[tuple[str, str, str], pd.DataFrame] = {}
baseline_artifacts: dict[tuple[str, str, str], dict] = {}

for dataset_tag, df_full in dataset_store.items():
    for split_name in PRIMARY_SPLITS:
        train_idx, test_idx = split_store[dataset_tag][split_name]
        for target_col in TARGET_COLS:
            run_key = (dataset_tag, split_name, target_col)
            artifact, metrics_df, diag_df = fit_baseline_for_split(df_full, train_idx, test_idx, target_col)
            metrics_df.insert(0, 'target_col', target_col)
            metrics_df.insert(0, 'split_name', split_name)
            metrics_df.insert(0, 'dataset_tag', dataset_tag)
            baseline_results.append(metrics_df)
            baseline_diags[run_key] = diag_df
            baseline_artifacts[run_key] = artifact
            print(f'Completed baseline run: {run_key}')

baseline_results_df = pd.concat(baseline_results, ignore_index=True)
display(
    baseline_results_df.sort_values(['dataset_tag', 'split_name', 'target_col', 'MAPE_pct', 'MAE']).reset_index(drop=True)
)

Completed baseline run: ('2026-05-28_001', 'price_stratified', 'price_egp_log')
Completed baseline run: ('2026-05-28_001', 'price_stratified', 'price_egp')
Completed baseline run: ('2026-05-28_001', 'make_model_grouped', 'price_egp_log')
Completed baseline run: ('2026-05-28_001', 'make_model_grouped', 'price_egp')
Completed baseline run: ('2026-06-03_008', 'price_stratified', 'price_egp_log')
Completed baseline run: ('2026-06-03_008', 'price_stratified', 'price_egp')
Completed baseline run: ('2026-06-03_008', 'make_model_grouped', 'price_egp_log')
Completed baseline run: ('2026-06-03_008', 'make_model_grouped', 'price_egp')


,dataset_tag,split_name,target_col,framework,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Mean_width_pct
0,2026-05-28_001,make_model_grouped,price_egp,LightGBM,"281,218.5915","748,156.3088",0.7938,24.8274,28.8322,43.6445,64.6917,64.6917,60.7306
1,2026-05-28_001,make_model_grouped,price_egp,WeightedEnsemble,"287,751.8501","705,248.3725",0.8168,28.9919,27.9366,41.1299,74.2680,74.2680,68.6562
2,2026-05-28_001,make_model_grouped,price_egp,XGBoost,"316,008.1100","703,749.0031",0.8175,34.4483,24.8708,35.5150,76.7137,76.7137,75.2509
3,2026-05-28_001,make_model_grouped,price_egp_log,LightGBM,"266,564.4535","733,126.4221",0.8020,22.7889,31.5880,47.5715,67.5508,67.5508,55.9983
4,2026-05-28_001,make_model_grouped,price_egp_log,WeightedEnsemble,"267,403.1391","727,924.2902",0.8048,23.9681,30.2101,45.5047,76.7137,76.7137,67.0749
5,2026-05-28_001,make_model_grouped,price_egp_log,XGBoost,"281,299.1735","734,938.9471",0.8010,26.2840,29.5901,42.5766,80.3307,80.3307,77.0150
6,2026-05-28_001,price_stratified,price_egp,WeightedEnsemble,"136,752.9526","481,807.9057",0.8930,14.8086,59.9075,75.9188,87.4068,87.4068,73.2532
7,2026-05-28_001,price_stratified,price_egp,XGBoost,"134,441.2527","469,595.9205",0.8983,14.8798,59.4192,74.9422,87.7923,87.7923,83.9819
8,2026-05-28_001,price_stratified,price_egp,LightGBM,"144,928.4067","511,601.4033",0.8793,15.2778,59.0594,75.5076,80.5962,80.5962,60.6431
9,2026-05-28_001,price_stratified,price_egp_log,WeightedEnsemble,"136,456.5038","486,716.5460",0.8908,14.2541,59.7533,76.4071,87.1755,87.1755,61.6411


## 7) Decision Table

Collapse the baseline matrix into the best configuration per dataset/split/target and identify the primary baseline recipe.

In [7]:
decision_table = (
    baseline_results_df
    .sort_values(['dataset_tag', 'split_name', 'target_col', 'MAPE_pct', 'MAE'])
    .groupby(['dataset_tag', 'split_name', 'target_col'], as_index=False)
    .first()
    .sort_values(['MAPE_pct', 'MAE'])
    .reset_index(drop=True)
)
display(decision_table)

selected_baseline = decision_table.iloc[0].to_dict()
selected_baseline

,dataset_tag,split_name,target_col,framework,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Mean_width_pct
0,2026-06-03_008,price_stratified,price_egp_log,WeightedEnsemble,"150,593.8633","605,558.6220",0.8875,11.3258,61.2400,76.7600,85.7600,85.7600,61.5454


{'dataset_tag': '2026-06-03_008',
 'split_name': 'price_stratified',
 'target_col': 'price_egp_log',
 'framework': 'WeightedEnsemble',
 'MAE': 150593.86325308098,
 'RMSE': 605558.621958141,
 'R2': 0.8875250419963173,
 'MAPE_pct': 11.32582524124563,
 'Within_10pct': 61.24000000000001,
 'Within_15pct': 76.75999999999999,
 'Coverage_80_pct': 85.76,
 'Coverage_90_pct': 85.76,
 'Mean_width_pct': 61.54542372219617}

## 8) Best-Run Per Make-Model Diagnostics

Inspect the per-make/model MAPE profile for the selected best baseline run before moving into feature ablations.

In [8]:
best_key = (
    selected_baseline['dataset_tag'],
    selected_baseline['split_name'],
    selected_baseline['target_col'],
)
print('Best baseline run key:', best_key)
print('Best framework:', selected_baseline['framework'])

print('Best 50 (lowest MAPE first):')
display(baseline_diags[best_key].sort_values('MAPE_pct', ascending=True).head(50))

Best baseline run key: ('2026-06-03_008', 'price_stratified', 'price_egp_log')
Best framework: WeightedEnsemble
Best 50 (lowest MAPE first):


,make,model,n_test,MAPE_pct
159,Audi,A3,8,2.8556
158,Chery,Tiggo 3,11,3.2885
157,Land Rover,Range Rover Velar,5,3.2924
156,MG,ZS,12,3.7747
155,Mercedes,GLC300,14,3.9318
154,MG,6,12,4.0859
153,Hyundai,Tucson,25,4.2910
152,Skoda,Kodiaq,24,4.7344
151,Chery,Tiggo 7,8,5.1422
150,Land Rover,Range Rover Evoque,8,5.2012


In [ ]:
worst_key = (
    selected_baseline['dataset_tag'],
    selected_baseline['split_name'],
    selected_baseline['target_col'],
)
print('Best baseline run key:', worst_key)
print('Best framework:', selected_baseline['framework'])

print('Worst 50 (highest MAPE first):')
display(baseline_diags[worst_key].sort_values('MAPE_pct', ascending=False).head(50))

Best baseline run key: ('2026-06-03_008', 'price_stratified', 'price_egp_log')
Best framework: WeightedEnsemble
Worst 50 (highest MAPE first):


,make,model,n_test,MAPE_pct
0,Chevrolet,Pickup,5,44.1606
1,Mazda,323,6,44.0812
2,Suzuki,Swift,8,37.0868
3,Peugeot,504,6,36.8888
4,Peugeot,405,9,31.3544
5,Honda,Civic,10,28.2366
6,Mercedes,200,8,27.1698
7,Fiat,128,11,22.8919
8,Mercedes,S500,5,22.3866
9,Mercedes,G63,6,21.7425


In [15]:
# Use the same config as the fast-path best run
cv_diag_df, cv_overall = run_kfold_per_make_model_cv(
    df_full,
    FEATURE_COLS,
    CAT_COLS,
    target_col,
    framework='XGBoost',
    params=XGB_BASE_PARAMS,
    n_splits=5,
    random_state=RANDOM_STATE,
    min_rows=5,
)

print('Overall CV metrics:')
for k, v in cv_overall.items():
    print(f'  {k}: {v:,.4f}')

Overall CV metrics:
  MAE: 155,669.4843
  RMSE: 509,000.4517
  R2: 0.9213
  MAPE_pct: 11.9933
  Within_10pct: 60.9458
  Within_15pct: 76.2922


In [13]:
print('\nCV per-make-model combos:', len(cv_diag_df))
print('\nBest 20 (lowest MAPE):')
display(cv_diag_df.sort_values('MAPE_pct', ascending=True).head(20))

print('\nWorst 20 (highest MAPE):')
display(cv_diag_df.sort_values('MAPE_pct', ascending=False).head(20))

cv_out_path = OUT_DIR / 'make_model_mape_cv.csv'
cv_diag_df.to_csv(cv_out_path, index=False)


CV per-make-model combos: 441

Best 20 (lowest MAPE):


,make,model,n,MAPE_pct,MAE,R2,mean_price
0,Volkswagen,Tayron,10,2.2118,"63,476.9650",-2.7254,"2,880,000.0000"
1,Jetour,X90,11,2.4873,"37,244.8794",0.7296,"1,501,363.6364"
2,Kaiyi,X3 Pro,5,2.6420,"25,040.3762",-0.6560,"964,000.0000"
3,Geely,GX3 Pro,8,2.8966,"23,605.4255",0.8701,"818,750.0000"
4,Cupra,Formentor,23,3.2614,"65,134.6621",0.7548,"2,133,435.1739"
5,BAIC,BJ30,12,3.3171,"56,578.5686",0.2065,"1,694,575.0000"
6,Jetour,X70 Plus,9,3.8635,"49,690.8412",-0.0929,"1,297,430.5556"
7,MG,7,10,3.9264,"65,394.1718",0.5894,"1,638,500.0000"
8,Kia,Xceed,23,3.9614,"49,337.5104",0.7609,"1,316,086.9565"
9,Chery,Arrizo 6,11,4.0103,"41,283.0292",0.0798,"1,029,274.5455"



Worst 20 (highest MAPE):


,make,model,n,MAPE_pct,MAE,R2,mean_price
440,Toyota,Cressida,5,89.4062,"122,735.1345",-0.2737,"162,000.0000"
439,Daewoo,Juliet,17,77.9824,"75,628.5449",-11.3872,"247,647.0588"
438,Volvo,XC90,8,65.4878,"972,370.4068",-0.2877,"2,821,875.0000"
437,Daewoo,Espero,7,57.8562,"326,185.0197",-0.1266,"401,428.5714"
436,Lada,2017,10,56.4155,"62,357.6623",-0.8003,"124,300.0000"
435,Fiat,132,7,54.6447,"44,500.1850",-396.9710,"80,714.2857"
434,Dodge,Charger,5,47.0392,"1,959,253.5840",0.3253,"4,096,000.0000"
433,Volkswagen,Beetle,6,46.6625,"156,422.0605",0.8130,"469,166.6667"
432,Changan,Benni,8,46.2583,"103,094.1914",-4.1714,"217,500.0000"
431,Citroën,Ax,5,42.3344,"35,182.6959",-4.0568,"93,000.0000"


## 9) Export Baseline Artifacts

Save the baseline summary tables, frozen splits, and best-run diagnostics for notebook `07b` to consume.

In [14]:
decision_table.to_csv(OUT_DIR / 'baseline_results_summary.csv', index=False)
baseline_results_df.to_csv(OUT_DIR / 'baseline_results_full.csv', index=False)
summary_df.to_csv(OUT_DIR / 'dataset_summary.csv', index=False)

split_export_rows = []
for dataset_tag, split_map in split_store.items():
    for split_name, (train_idx, test_idx) in split_map.items():
        split_export_rows.append(
            {
                'dataset_tag': dataset_tag,
                'split_name': split_name,
                'train_indices': ','.join(map(str, train_idx.tolist())),
                'test_indices': ','.join(map(str, test_idx.tolist())),
            }
        )
split_export_df = pd.DataFrame(split_export_rows)
split_export_df.to_csv(OUT_DIR / 'frozen_splits.csv', index=False)

best_diag_path = OUT_DIR / f"best_make_model_mape__{best_key[0]}__{best_key[1]}__{best_key[2]}.csv"
baseline_diags[best_key].to_csv(best_diag_path, index=False)

print('Saved baseline artifacts to:', OUT_DIR)

Saved baseline artifacts to: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07a_dataset_baselines
